In [1]:
import os

In [2]:
%pwd

'c:\\Users\\Ayush\\OneDrive\\Desktop\\new folder\\End-to-End-machine-learning-project-with-mlflow\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\Ayush\\OneDrive\\Desktop\\new folder\\End-to-End-machine-learning-project-with-mlflow'

In [5]:
os.environ["MLFLOW_TRACKING_URI"] = "https://dagshub.com/ayush9492/End-to-End-machine-learning-project-with-mlflow.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"] = "ayush9492"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "b37174b788eb480a05a34451a642c135e6893057"

In [16]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    test_data_path: Path
    model_path: Path
    all_params: dict
    metric_file_name: Path
    target_column: str
    mlflow_uri: str


In [17]:
from mlProject.constants import *
from mlProject.utils.common import read_yaml, create_directories, save_json

In [18]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    
    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation
        params = self.params.ElasticNet
        schema =  self.schema.TARGET_COLUMN

        create_directories([config.root_dir])

        model_evaluation_config = ModelEvaluationConfig(
            root_dir=config.root_dir,
            test_data_path=config.test_data_path,
            model_path = config.model_path,
            all_params=params,
            metric_file_name = config.metric_file_name,
            target_column = schema.name,
            mlflow_uri="https://dagshub.com/ayush9492/End-to-End-machine-learning-project-with-mlflow.mlflow",
           
        )

        return model_evaluation_config


In [19]:
! pip install mlflow
import os
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from urllib.parse import urlparse
import mlflow
import mlflow.sklearn
import numpy as np
import joblib


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [20]:
class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config

    
    def eval_metrics(self,actual, pred):
        rmse = np.sqrt(mean_squared_error(actual, pred))
        mae = mean_absolute_error(actual, pred)
        r2 = r2_score(actual, pred)
        return rmse, mae, r2
    


    def log_into_mlflow(self):

        test_data = pd.read_csv(self.config.test_data_path)
        model = joblib.load(self.config.model_path)

        test_x = test_data.drop([self.config.target_column], axis=1)
        test_y = test_data[[self.config.target_column]]


        mlflow.set_registry_uri(self.config.mlflow_uri)
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme


        with mlflow.start_run():

            predicted_qualities = model.predict(test_x)

            (rmse, mae, r2) = self.eval_metrics(test_y, predicted_qualities)
            
            # Saving metrics as local
            scores = {"rmse": rmse, "mae": mae, "r2": r2}
            save_json(path=Path(self.config.metric_file_name), data=scores)

            mlflow.log_params(self.config.all_params)

            mlflow.log_metric("rmse", rmse)
            mlflow.log_metric("r2", r2)
            mlflow.log_metric("mae", mae)


            # Model registry does not work with file store
            if tracking_url_type_store != "file":

                # Register the model
                # There are other ways to use the Model Registry, which depends on the use case,
                # please refer to the doc for more information:
                # https://mlflow.org/docs/latest/model-registry.html#api-workflow
                mlflow.sklearn.log_model(model, "model", registered_model_name="ElasticnetModel")
            else:
                mlflow.sklearn.log_model(model, "model")

    


In [21]:
try:
    config = ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation_config = ModelEvaluation(config=model_evaluation_config)
    model_evaluation_config.log_into_mlflow()
except Exception as e:
    raise e

[2026-03-29 06:29:05,661: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-03-29 06:29:05,681: INFO: common: yaml file: params.yaml loaded successfully]
[2026-03-29 06:29:05,686: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-03-29 06:29:05,689: INFO: common: created directory at: artifacts]
[2026-03-29 06:29:05,693: INFO: common: created directory at: artifacts/model_evaluation]
[2026-03-29 06:29:06,614: INFO: common: json file saved at: artifacts\model_evaluation\metric.json]


2026/03/29 06:29:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/29 06:29:08 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'ElasticnetModel' already exists. Creating a new version of this model...
2026/03/29 06:29:18 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: ElasticnetModel, version 2
Created version '2' of model 'ElasticnetModel'.


🏃 View run big-stoat-317 at: https://dagshub.com/ayush9492/End-to-End-machine-learning-project-with-mlflow.mlflow/#/experiments/0/runs/f5b1ca09c6f0495b8719fc5e44f8f4ed
🧪 View experiment at: https://dagshub.com/ayush9492/End-to-End-machine-learning-project-with-mlflow.mlflow/#/experiments/0
